<a href="https://colab.research.google.com/github/gloriabee/DEMovies/blob/main/silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import duckdb

In [ ]:
DRIVE_DB= "/content/drive/MyDrive/data_engineering/movies.duckdb"
LOCAL_DB="/content/movies.duckdb"

In [ ]:
print("Exists:",os.path.exists(DRIVE_DB))

In [ ]:
if os.path.exists(DRIVE_DB):
  print("Drive DB size:",os.path.getsize(DRIVE_DB))


In [ ]:
con=duckdb.connect(DRIVE_DB)

In [ ]:
con.execute("SHOW TABLES").fetchdf()

In [ ]:
movies_df=con.execute("""
SELECT *
FROM movies_bronze
""").fetchdf()

In [ ]:
movies_df.head()

## Start Silver Transformations

### remove irrelevant columns

In [ ]:
movies_df.columns

In [ ]:
movies_df=movies_df.drop(columns=['IMDbLink','Title_IMDb_Link'])

In [ ]:
movies_df.head()

### checking missing values

In [ ]:
movies_df.info()

In [ ]:
for col in ["directors","writers","genres"]:
  print(col, (movies_df[col].str.strip()=="").sum())

In [ ]:
print("Duplicate tconst:",movies_df["tconst"].duplicated().sum())

### Checking Logical Integrity

In [ ]:
invalid_rating=movies_df[
    (movies_df['averageRating']>10) |
    (movies_df['averageRating']<0)
]

print("Invalid rating :",len(invalid_rating))

In [ ]:
invalid_runtime=movies_df[
    movies_df['runtimeMinutes']<=0
]

print("Invalid rows:",len(invalid_runtime))

In [ ]:
invalid_year=movies_df[
    (movies_df['startYear']>2026) |
    (movies_df['startYear']<1900)
]
print("Invalid rows: ",len(invalid_year))


In [ ]:
invalid_votes=movies_df[movies_df['numVotes']<0]

print("Invalid rows:",len(invalid_votes))

In [ ]:
movies_df.head(2)

### Silver Transformation (Movie Directors)

In [ ]:
movies_directors=movies_df[["tconst","directors"]].copy()

In [ ]:
movies_directors["director"]=movies_directors["directors"].apply(
    lambda x: x.split(',') if "," in x else [x]
)

In [ ]:
movies_directors= movies_directors.explode("director")

In [ ]:
movies_directors["director"]=movies_directors["director"].str.strip()

In [ ]:
movies_directors= movies_directors.drop(columns="directors")

In [ ]:
movies_directors

### Silver Transformation (Movie Writers)

In [45]:
movies_writers=movies_df[["tconst","writers"]].copy()

In [47]:
movies_writers["writer"]=movies_writers["writers"].apply(
    lambda x: x.split(',') if "," in x else [x]
)

In [48]:
movies_writers= movies_writers.explode("writer")

In [50]:
movies_writers["writer"]=movies_writers["writer"].str.strip()

In [51]:
movies_writers= movies_writers.drop(columns="writers")

In [53]:
movies_writers

,tconst,writer
0,tt0111161,Stephen King
0,tt0111161,Frank Darabont
1,tt0068646,Mario Puzo
1,tt0068646,Francis Ford Coppola
2,tt0468569,Jonathan Nolan
...,...,...
4998,tt5155780,Justin Lader
4999,tt0077394,Harvey Bernhard
4999,tt0077394,David Seltzer
4999,tt0077394,Stanley Mann


### Silver Transformation (Movie Genres)

In [54]:
movies_genres=movies_df[["tconst","genres"]].copy()

In [57]:
movies_genres["genre"]=movies_genres["genres"].apply(
    lambda x: x.split(',') if "," in x else [x]
)

In [58]:
movies_genres= movies_genres.explode("genre")

In [59]:
movies_genres["genre"]=movies_genres["genre"].str.strip()

In [62]:
movies_genres= movies_genres.drop(columns="genres")

In [63]:
movies_genres

,tconst,genre
0,tt0111161,Drama
1,tt0068646,Crime
1,tt0068646,Drama
2,tt0468569,Crime
2,tt0468569,Thriller
...,...,...
4997,tt1477837,Comedy
4998,tt5155780,Drama
4998,tt5155780,Romance
4998,tt5155780,Sci-Fi


### Load cleansed data in silver duckdb tables


*   movies_silver
*   movies_directors_silver
*   movies_writers_silver
*   movies_genres_silver



In [66]:
movies_df=movies_df.drop(columns=["directors","writers","genres"])

In [67]:
con.register(f"movies_silver_tmp",movies_df)
con.execute("""
  CREATE OR REPLACE TABLE movies_silver AS
  SELECT *
  FROM movies_silver_tmp
""")
con.unregister("movies_silver_tmp")

In [70]:
con.register(f"movies_directors_tmp",movies_directors)
con.execute("""
  CREATE OR REPLACE TABLE movies_directors_silver AS
  SELECT *
  FROM movies_directors_tmp
""")
con.unregister("movies_directors_tmp")

In [72]:
con.register(f"movies_writers_tmp",movies_writers)
con.execute("""
  CREATE OR REPLACE TABLE movies_writers_silver AS
  SELECT *
  FROM movies_writers_tmp
""")
con.unregister("movies_writers_tmp")

In [74]:
con.register(f"movies_genres_tmp",movies_genres)
con.execute("""
  CREATE OR REPLACE TABLE movies_genres_silver AS
  SELECT *
  FROM movies_genres_tmp
""")
con.unregister("movies_genres_tmp")

In [75]:
con.execute("SHOW TABLES").fetchdf()

,name
0,movies_bronze
1,movies_directors_silver
2,movies_genres_silver
3,movies_silver
4,movies_writers_silver


In [77]:
# print("Local DB size:", os.path.getsize(LOCAL_DB))
print("Drive DB size:", os.path.getsize(DRIVE_DB))

Drive DB size: 1060864
